# Pilhas: conceito e implementação — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

A aula passada terminou com uma pista: inserir e remover nas pontas da lista ficou barato, e uma
estrutura que *só* faz isso já tem nome. Este tutorial retoma o programa `balanco.c` visto hoje — que
decide se parênteses, colchetes e chaves fecham na ordem certa — e segue o caminho dos slides: o que é
uma pilha, a pilha num vetor e seus dois erros silenciosos, o vetor que cresce, a pilha numa lista e o
custo de cada versão. Depois de cada bloco há um exercício com teste automático; no fim vêm as tarefas
de modificação e o desafio.

## Objetivos

Ao final deste tutorial você será capaz de:

- Descrever uma pilha pelas suas quatro operações e reconhecer problemas que pedem uma
- Implementar a pilha num vetor, com o topo em `v[n-1]`, conferindo pilha vazia e pilha cheia
- Explicar por que um vetor que **dobra** custa um número constante de cópias por elemento, e um que
  cresce de um em um, não
- Implementar a pilha numa lista simples, com o topo no primeiro nó
- Esconder a implementação atrás de `pilha.h` e trocar vetor por lista sem mexer em quem usa a pilha
- Usar pilhas para inverter, converter de base e avaliar expressões pós-fixas

In [ ]:
# Confira se o gcc está disponível e defina a função que testa os exercícios
!gcc --version | head -1

import subprocess

def testa(comando, esperado, entrada=None, tempo=20):
    '''Roda o comando e compara a saída com o esperado.'''
    try:
        r = subprocess.run(comando, shell=True, capture_output=True, text=True,
                           input=entrada, timeout=tempo)
    except subprocess.TimeoutExpired:
        print(f'Demorou mais de {tempo} s: algum laço está fazendo trabalho à toa?')
        return
    limpa = lambda t: '\n'.join(l.rstrip() for l in t.strip().split('\n'))
    saida, esperado = limpa(r.stdout), limpa(esperado)
    if r.returncode != 0:
        print(f'O programa terminou com erro (status {r.returncode}; '
              '-11 ou 139 = segmentation fault).')
        if r.stderr:
            print(r.stderr[:1500])
    if saida == esperado:
        print('OK!')
    else:
        print('Saída esperada:')
        print(esperado)
        print('Saída obtida:')
        print(saida)

## O programa da aula

Recompile o programa condutor **exatamente como saiu da aula**. Antes de executar, escreva a saída que
você espera: `sim` ou `nao` para cada uma das cinco cadeias — inclusive a vazia.

In [ ]:
%%writefile balanco.c
/* balanco.c --- os parenteses, colchetes e chaves fecham na ordem certa? */
#include <stdio.h>

#define MAX 100

typedef struct {
    char v[MAX];   /* v[0..n-1]: o topo e v[n-1] */
    int n;         /* quantos elementos ha na pilha */
} Pilha;

void empilha(Pilha *p, char c)
{
    p->v[p->n] = c;
    p->n++;
}

char desempilha(Pilha *p)
{
    p->n--;
    return p->v[p->n];
}

int vazia(const Pilha *p)
{
    return p->n == 0;
}

/* 1 se o par (abre, fecha) combina: ( com ), [ com ], { com } */
int combina(char abre, char fecha)
{
    return (abre == '(' && fecha == ')') ||
           (abre == '[' && fecha == ']') ||
           (abre == '{' && fecha == '}');
}

int balanceada(const char *s)
{
    Pilha p = {.n = 0};
    for (int i = 0; s[i] != '\0'; i++) {
        char c = s[i];
        if (c == '(' || c == '[' || c == '{')
            empilha(&p, c);
        else if (c == ')' || c == ']' || c == '}') {
            if (vazia(&p))
                return 0;          /* fechou sem ter aberto */
            if (!combina(desempilha(&p), c))
                return 0;          /* fechou o par errado */
        }
    }
    return vazia(&p);              /* sobrou algum aberto? */
}

int main(void)
{
    const char *testes[] = {
        "v[f(x)] = {1, 2}",
        "([)]",
        "(()",
        "())",
        "",
    };
    for (int i = 0; i < 5; i++)
        printf("%-18s %s\n", testes[i], balanceada(testes[i]) ? "sim" : "nao");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra balanco.c -o balanco && ./balanco

## 1. O que é uma pilha

Uma **pilha** é uma sequência em que inserções e remoções acontecem só numa das pontas, o *topo*: o
último a entrar é o primeiro a sair (*LIFO*, *last in, first out*). Quatro operações bastam:

| operação | o que faz |
|---|---|
| `empilha(p, x)` | põe `x` no topo |
| `desempilha(p)` | tira e devolve o topo |
| `topo(p)` | devolve o topo sem tirar |
| `vazia(p)` | a pilha não tem nada? |

Em `balanco.c`, a pilha guarda os abertos que ainda não fecharam. Cada fechamento só precisa conferir o
**topo** — o aberto mais recente sem par —, e os de baixo esperam a vez. Três coisas podem dar errado:
fechar o par errado (`([)]`), sobrar aberto no fim (`(()`) e fechar sem ter aberto (`())`).

A pilha de execução da aula de recursão é o mesmo mecanismo: cada chamada empilha um registro de
ativação, cada `return` desempilha, e por isso as chamadas terminam na ordem inversa em que começaram.

### Exercício 1 — inverter uma cadeia

A propriedade mais direta da pilha: o que entra numa ordem sai na ordem inversa. Escreva `inverte`, que
inverte `s` no lugar usando a `Pilha` da aula — empilhe todos os caracteres e desempilhe-os de volta em
`s`.

In [ ]:
%%writefile ex1.c
#include <stdio.h>

#define MAX 100

typedef struct {
    char v[MAX];
    int n;
} Pilha;

void empilha(Pilha *p, char c)
{
    p->v[p->n] = c;
    p->n++;
}

char desempilha(Pilha *p)
{
    p->n--;
    return p->v[p->n];
}

int vazia(const Pilha *p)
{
    return p->n == 0;
}

/* Exercicio 1: inverte s no lugar usando uma Pilha (s tem menos de MAX
   caracteres). TODO: implemente inverte. */
void inverte(char *s)
{
    /* TODO: implemente aqui */
    (void) s;
}

int main(void)
{
    char a[] = "pilha", b[] = "", c[] = "a", d[] = "roma me tem amor",
         e[] = "[)](";
    inverte(a);
    inverte(b);
    inverte(c);
    inverte(d);
    inverte(e);
    printf("[%s]\n[%s]\n[%s]\n[%s]\n[%s]\n", a, b, c, d, e);
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex1.c -o ex1
testa('./ex1', '''[ahlip]
[]
[a]
[roma met em amor]
[(])[]''')

## 2. Pilha num vetor

No vetor, o fundo é `v[0]` e o topo é `v[n-1]`; `n` conta quantos elementos há. Empilhar escreve em
`v[n]` e incrementa `n`; desempilhar decrementa `n` e lê `v[n]`. Nenhuma operação tem laço.

O preço da simplicidade são dois erros que o C não confere por você:

- desempilhar a pilha **vazia** (*underflow*) lê `v[-1]`, fora do vetor;
- empilhar na pilha **cheia** (*overflow*) escreve `v[MAX]`, fora do vetor.

A célula abaixo gera `balanco_errada.c`, que esquece de conferir a pilha vazia antes de desempilhar.
Preveja a saída antes de executar — e depois rode a versão com `-fsanitize=address`.

In [ ]:
!python3 -c "s=open('balanco.c').read(); s=s.replace('            if (vazia(&p))\n                return 0;          /* fechou sem ter aberto */\n', ''); open('balanco_errada.c','w').write(s)"
!grep -n -A3 "else if" balanco_errada.c
!gcc -Wall -Wextra balanco_errada.c -o balanco_errada && ./balanco_errada
!gcc -g -fsanitize=address balanco_errada.c -o balanco_errada-asan && ./balanco_errada-asan 2>&1 | head -14

A saída sem o *sanitizer* é **idêntica** à de `balanco.c`: em `"())"`, o segundo `)` leu `v[-1]`, um
lixo qualquer, `combina` devolveu 0 e a resposta saiu `nao` — a certa, por acaso. O ASan mostra o que
realmente aconteceu: **stack-buffer-underflow**, uma leitura logo antes da variável `p`, dentro de
`desempilha`. Certo por sorte é errado: com outro compilador, o lixo pode ser um `(`.

E se a pilha encher? `transborda.c` empilha `k` vezes `'('` e depois tenta desempilhar `k` vezes, com
`MAX` igual a 100. Para `k = 101`, onde vai parar o último `'('`? Que valor de `n` você espera?

In [ ]:
%%writefile transborda.c
/* transborda.c --- a pilha de balanco.c com mais de MAX abertos */
#include <stdio.h>

#define MAX 100

typedef struct {
    char v[MAX];   /* v[0..n-1]: o topo e v[n-1] */
    int n;         /* quantos elementos ha na pilha */
} Pilha;

void empilha(Pilha *p, char c)
{
    p->v[p->n] = c;
    p->n++;
}

char desempilha(Pilha *p)
{
    p->n--;
    return p->v[p->n];
}

int vazia(const Pilha *p)
{
    return p->n == 0;
}

int main(void)
{
    int niveis[] = {99, 100, 101, 150};
    for (int i = 0; i < 4; i++) {
        Pilha p = {.n = 0};
        int k = niveis[i], fechou = 0;
        for (int j = 0; j < k; j++)          /* k vezes '(' ... */
            empilha(&p, '(');
        printf("%3d abertos: n = %3d", k, p.n);
        while (!vazia(&p) && fechou < k) {   /* ... e k vezes ')' */
            desempilha(&p);
            fechou++;
        }
        printf(", fechou %3d -> %s\n", fechou,
               fechou == k && vazia(&p) ? "sim" : "nao");
    }
    return 0;
}

In [ ]:
!gcc -Wall -Wextra transborda.c -o transborda && ./transborda
!gcc -g -fsanitize=address transborda.c -o transborda-asan && ./transborda-asan

No registro `Pilha`, o campo `n` vem logo depois dos 100 bytes de `v`. O 101º `empilha` escreve `'('` —
código 40 — em `v[100]`, que é o byte menos significativo de `n` (x86-64 e ARM guardam esse byte
primeiro). `n` valia 100 e passa a valer 40; o `n++` seguinte faz 41. Os próximos `'('` são escritos
por cima dos antigos, *dentro* do vetor. E o *sanitizer* não reclama: a escrita caiu dentro do objeto
`p`, e ele só vigia as bordas dos objetos.

Confira a disposição do registro:

In [ ]:
%%writefile layout.c
#include <stdio.h>
#include <stddef.h>

#define MAX 100

typedef struct {
    char v[MAX];
    int n;
} Pilha;

int main(void)
{
    printf("sizeof(Pilha) = %zu\n", sizeof(Pilha));
    printf("offsetof v = %zu, offsetof n = %zu\n",
           offsetof(Pilha, v), offsetof(Pilha, n));
    printf("'(' = %d\n", '(');
    return 0;
}

In [ ]:
!gcc -Wall -Wextra layout.c -o layout && ./layout

### Exercício 2 — uma pilha que avisa

Para que quem usa a pilha *possa* conferir, as operações precisam contar o que aconteceu. Implemente:

- `int empilha(Pilha *p, char c)`: devolve 1 se empilhou e 0 se a pilha estava cheia;
- `int desempilha(Pilha *p, char *c)`: devolve 1 e o topo em `*c`, ou 0 se a pilha estava vazia.

Em caso de falha, a pilha não pode mudar. `MAX` é 4 de propósito; a `balanceada` já usa as novas
funções e responde `cheia` quando a cadeia é aninhada demais. O teste roda com `-fsanitize=address`.

In [ ]:
%%writefile ex2.c
#include <stdio.h>

#define MAX 4   /* pequeno de proposito */

typedef struct {
    char v[MAX];
    int n;
} Pilha;

/* Exercicio 2: empilha devolve 1 se conseguiu e 0 se a pilha esta cheia;
   desempilha devolve 1 e o topo em *c, ou 0 se a pilha esta vazia.
   Em caso de falha, a pilha nao pode mudar.
   TODO: implemente empilha e desempilha. */
int empilha(Pilha *p, char c)
{
    /* TODO: implemente aqui */
    (void) p;
    (void) c;
    return 0;
}

int desempilha(Pilha *p, char *c)
{
    /* TODO: implemente aqui */
    (void) p;
    (void) c;
    return 0;
}

int combina(char abre, char fecha)
{
    return (abre == '(' && fecha == ')') ||
           (abre == '[' && fecha == ']') ||
           (abre == '{' && fecha == '}');
}

/* 1 = balanceada, 0 = nao, -1 = aninhada demais para a pilha */
int balanceada(const char *s)
{
    Pilha p = {.n = 0};
    char topo;
    for (int i = 0; s[i] != '\0'; i++) {
        char c = s[i];
        if (c == '(' || c == '[' || c == '{') {
            if (!empilha(&p, c))
                return -1;
        } else if (c == ')' || c == ']' || c == '}') {
            if (!desempilha(&p, &topo) || !combina(topo, c))
                return 0;
        }
    }
    return p.n == 0;
}

int main(void)
{
    Pilha p = {.n = 0};
    char c = '?';
    int r = desempilha(&p, &c);
    printf("desempilha na vazia: %d (n = %d)\n", r, p.n);
    printf("empilha 4 vezes:");
    for (int i = 0; i < 4; i++)
        printf(" %d", empilha(&p, "abcd"[i]));
    r = empilha(&p, 'e');
    printf("\nempilha na cheia: %d (n = %d)\n", r, p.n);
    r = desempilha(&p, &c);
    printf("desempilha: %d '%c'\n", r, c);

    const char *testes[] = {"([])", "(((())))", "((((()))))", "())", "([)]", "(()"};
    const char *nome[] = {"nao", "sim", "cheia"};
    for (int i = 0; i < 6; i++) {
        int b = balanceada(testes[i]);
        printf("%-18s %s\n", testes[i], nome[b == -1 ? 2 : b]);
    }
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex2.c -o ex2
testa('./ex2', '''desempilha na vazia: 0 (n = 0)
empilha 4 vezes: 1 1 1 1
empilha na cheia: 0 (n = 4)
desempilha: 1 'd'
([])               sim
(((())))           sim
((((()))))         cheia
())                nao
([)]               nao
(()                nao''')

## 3. Quando o vetor enche

Um limite fixo raramente é a resposta certa. A saída é guardar a capacidade `cap` ao lado de `n` e
crescer com `realloc` quando `n == cap`. Mas crescer **quanto**?

- somando um (`cap + 1`), o vetor cresce a cada `empilha`; se cada crescimento copiar tudo, são
  $1 + 2 + \cdots + (n-1) = n(n-1)/2$ cópias;
- dobrando (`cap * 2`), cada crescimento copia metade do novo vetor, mas depois dele cabem outros
  tantos `empilha` sem cópia nenhuma: $1 + 2 + 4 + \cdots + n/2 < n$ cópias no total.

`crescimento.c` faz o vetor crescer "na mão" — aloca, copia, libera —, que é o que o `realloc` precisa
fazer quando não consegue aumentar o bloco no lugar, e conta as cópias.

In [ ]:
%%writefile crescimento.c
/* crescimento.c --- empilhar n valores num vetor que cresce de 1 em 1
   ou que dobra quando enche: quantas copias e quanto tempo.
   Aqui o vetor cresce "na mao" (aloca, copia, libera), que e o que o
   realloc precisa fazer quando nao consegue aumentar o bloco no lugar. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

static double agora(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

/* troca v por um bloco de nova_cap elementos com os tam primeiros copiados */
static int *cresce(int *v, int tam, int nova_cap, long long *copias)
{
    int *novo = malloc(nova_cap * sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    memcpy(novo, v, tam * sizeof *v);
    *copias += tam;
    free(v);
    return novo;
}

/* empilha n valores; dobra != 0 escolhe a estrategia; devolve as copias */
long long empilha_n(int n, int dobra)
{
    int cap = 1, tam = 0;
    int *v = malloc(cap * sizeof *v);
    long long copias = 0;
    if (v == NULL) {
        perror("malloc");
        exit(1);
    }
    for (int i = 0; i < n; i++) {
        if (tam == cap) {
            int nova = dobra ? 2 * cap : cap + 1;
            v = cresce(v, tam, nova, &copias);
            cap = nova;
        }
        v[tam] = i;
        tam++;
    }
    free(v);
    return copias;
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 50000;
    double t0 = agora();
    long long c1 = empilha_n(n, 0);
    double t1 = agora();
    long long c2 = empilha_n(n, 1);
    double t2 = agora();
    printf("n = %d: +1 copias %lld (%.4f s)  x2 copias %lld (%.4f s)\n",
           n, c1, t1 - t0, c2, t2 - t1);
    return 0;
}

In [ ]:
# Visualização: cópias e tempo para empilhar n valores, somando um x dobrando
import matplotlib.pyplot as plt

!gcc -O2 -Wall -Wextra crescimento.c -o crescimento

ns, c1, t1, c2, t2 = [], [], [], [], []
for n in [10000, 20000, 40000, 60000, 80000]:
    s = subprocess.run(['./crescimento', str(n)], capture_output=True, text=True).stdout
    print(s.strip())
    p = s.replace('(', ' ').replace(')', ' ').split()
    # n = N: +1 copias C1 T1 s  x2 copias C2 T2 s
    ns.append(n)
    c1.append(int(p[5])); t1.append(float(p[6]))
    c2.append(int(p[10])); t2.append(float(p[11]))

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(ns, c1, 'o-', label='cap + 1')
a.plot(ns, c2, 's-', label='cap * 2')
a.set_yscale('log'); a.set_xlabel('n'); a.set_ylabel('elementos copiados (log)')
a.set_title('Cópias'); a.legend(); a.grid(alpha=0.3)
b.plot(ns, t1, 'o-', label='cap + 1')
b.plot(ns, t2, 's-', label='cap * 2')
b.set_xlabel('n'); b.set_ylabel('tempo (s)')
b.set_title('Tempo'); b.legend(); b.grid(alpha=0.3)
plt.show()

O eixo das cópias está em escala logarítmica: a curva de `cap + 1` é a parábola $n^2/2$; a de
`cap * 2` fica abaixo de $2n$. Dividido entre todos os `empilha`, o custo do dobro é constante por
operação — o *custo amortizado*: um `empilha` raro é caro, a média não. O preço é memória: logo depois
de dobrar, metade do vetor está vazia.

O `realloc` de verdade às vezes consegue aumentar o bloco no lugar e não copia nada — mas a norma não
promete isso, e um programa não deve depender da sorte.

### Exercício 3 — o vetor que dobra

Implemente `empilha`: se a pilha estiver cheia, dobre a capacidade com `realloc` (conferindo o `NULL`)
e só então empilhe. O teste empilha um milhão de valores a partir de `cap = 1` e imprime cada nova
capacidade. Quantos crescimentos você espera?

In [ ]:
%%writefile ex3.c
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    int *v;
    int n;     /* quantos ha */
    int cap;   /* quantos cabem */
} Pilha;

void inicia(Pilha *p)
{
    p->cap = 1;
    p->n = 0;
    p->v = malloc(p->cap * sizeof *p->v);
    if (p->v == NULL) { perror("malloc"); exit(1); }
}

/* Exercicio 3: se a pilha estiver cheia, dobre a capacidade com realloc
   (e confira o NULL) antes de empilhar.
   TODO: implemente empilha. */
void empilha(Pilha *p, int x)
{
    /* TODO: implemente aqui */
    (void) p;
    (void) x;
}

int desempilha(Pilha *p)
{
    p->n--;
    return p->v[p->n];
}

int main(void)
{
    Pilha p;
    inicia(&p);
    int ultima = p.cap, crescimentos = 0;
    printf("cap: %d", p.cap);
    for (int i = 0; i < 1000000; i++) {
        empilha(&p, i);
        if (p.cap != ultima) {
            printf(" %d", p.cap);
            ultima = p.cap;
            crescimentos++;
        }
    }
    printf("\ncrescimentos: %d\n", crescimentos);
    long long soma = 0;
    int quantos = 0;
    while (p.n > 0) {
        soma += desempilha(&p);
        quantos++;
    }
    printf("desempilhados: %d (soma %lld)\n", quantos, soma);
    free(p.v);
    return 0;
}

In [ ]:
!gcc -O2 -Wall -Wextra ex3.c -o ex3
testa('./ex3', '''cap: 1 2 4 8 16 32 64 128 256 512 1024 2048 4096 8192 16384 32768 65536 131072 262144 524288 1048576
crescimentos: 20
desempilhados: 1000000 (soma 499999500000)''', tempo=5)

## 4. Pilha numa lista

Na lista simples, as duas operações baratas eram **inserir no início** e **remover o primeiro**. É tudo
o que uma pilha precisa: o topo é o primeiro nó. Não há `fim`, não há `ant`, não há laço — e não há
`cap`: a pilha só enche quando acaba a memória do processo. O *underflow* continua possível, por isso
`desempilha` confere a pilha vazia.

Para trocar uma implementação pela outra sem mexer em quem usa a pilha, as operações vão para um
cabeçalho, `pilha.h`. Ele só **declara** `struct pilha`; os campos ficam em cada `.c`. Quem inclui
`pilha.h` pode guardar um `Pilha *`, mas não consegue escrever `p->n` nem `sizeof(Pilha)` — e por isso
nunca depende de como a pilha é feita.

In [ ]:
%%writefile pilha.h
/* pilha.h --- o que o usuario da pilha precisa saber, e nada mais */
#ifndef PILHA_H
#define PILHA_H

typedef struct pilha Pilha;   /* os campos ficam no .c */

Pilha *pilha_cria(void);
void pilha_libera(Pilha *p);

void empilha(Pilha *p, int x);
int desempilha(Pilha *p);     /* tira e devolve o topo */
int topo(const Pilha *p);     /* devolve o topo sem tirar */
int vazia(const Pilha *p);

#endif

In [ ]:
%%writefile pilha_vetor.c
/* pilha_vetor.c --- pilha num vetor que dobra quando enche */
#include <stdio.h>
#include <stdlib.h>
#include "pilha.h"

struct pilha {
    int *v;    /* v[0..n-1]: o topo e v[n-1] */
    int n;     /* quantos elementos ha */
    int cap;   /* quantos cabem em v */
};

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

Pilha *pilha_cria(void)
{
    Pilha *p = aloca(NULL, sizeof *p);
    p->n = 0;
    p->cap = 4;
    p->v = aloca(NULL, p->cap * sizeof *p->v);
    return p;
}

void pilha_libera(Pilha *p)
{
    free(p->v);
    free(p);
}

void empilha(Pilha *p, int x)
{
    if (p->n == p->cap) {             /* cheio: dobra a capacidade */
        p->cap *= 2;
        p->v = aloca(p->v, p->cap * sizeof *p->v);
    }
    p->v[p->n] = x;
    p->n++;
}

int desempilha(Pilha *p)
{
    if (p->n == 0) {
        fprintf(stderr, "desempilha: pilha vazia\n");
        exit(1);
    }
    p->n--;
    return p->v[p->n];
}

int topo(const Pilha *p)
{
    if (p->n == 0) {
        fprintf(stderr, "topo: pilha vazia\n");
        exit(1);
    }
    return p->v[p->n - 1];
}

int vazia(const Pilha *p)
{
    return p->n == 0;
}

In [ ]:
%%writefile pilha_lista.c
/* pilha_lista.c --- pilha numa lista encadeada: o topo e o primeiro no */
#include <stdio.h>
#include <stdlib.h>
#include "pilha.h"

typedef struct no {
    int valor;
    struct no *prox;   /* o de baixo, ou NULL */
} No;

struct pilha {
    No *topo;          /* primeiro no, ou NULL */
};

Pilha *pilha_cria(void)
{
    Pilha *p = malloc(sizeof *p);
    if (p == NULL) {
        perror("malloc");
        exit(1);
    }
    p->topo = NULL;
    return p;
}

void pilha_libera(Pilha *p)
{
    while (!vazia(p))
        desempilha(p);
    free(p);
}

void empilha(Pilha *p, int x)          /* insere no inicio */
{
    No *novo = malloc(sizeof *novo);
    if (novo == NULL) {
        perror("malloc");
        exit(1);
    }
    novo->valor = x;
    novo->prox = p->topo;
    p->topo = novo;
}

int desempilha(Pilha *p)               /* remove o primeiro */
{
    if (p->topo == NULL) {
        fprintf(stderr, "desempilha: pilha vazia\n");
        exit(1);
    }
    No *velho = p->topo;
    int x = velho->valor;
    p->topo = velho->prox;
    free(velho);
    return x;
}

int topo(const Pilha *p)
{
    if (p->topo == NULL) {
        fprintf(stderr, "topo: pilha vazia\n");
        exit(1);
    }
    return p->topo->valor;
}

int vazia(const Pilha *p)
{
    return p->topo == NULL;
}

In [ ]:
%%writefile balanco_tad.c
/* balanco_tad.c --- o mesmo teste de balanco.c, usando so pilha.h
   gcc balanco_tad.c pilha_vetor.c   ou   gcc balanco_tad.c pilha_lista.c */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "pilha.h"

int combina(int abre, int fecha)
{
    return (abre == '(' && fecha == ')') ||
           (abre == '[' && fecha == ']') ||
           (abre == '{' && fecha == '}');
}

int balanceada(const char *s)
{
    Pilha *p = pilha_cria();
    int ok = 1;
    for (int i = 0; ok && s[i] != '\0'; i++) {
        char c = s[i];
        if (c == '(' || c == '[' || c == '{')
            empilha(p, c);
        else if (c == ')' || c == ']' || c == '}')
            ok = !vazia(p) && combina(desempilha(p), c);
    }
    ok = ok && vazia(p);
    pilha_libera(p);
    return ok;
}

int main(void)
{
    const char *testes[] = {"v[f(x)] = {1, 2}", "([)]", "(()", "())", ""};
    for (int i = 0; i < 5; i++)
        printf("%-18s %s\n", testes[i], balanceada(testes[i]) ? "sim" : "nao");

    int k = 100000;                    /* k abertos seguidos de k fechados */
    char *s = malloc(2 * k + 1);
    if (s == NULL) {
        perror("malloc");
        return 1;
    }
    memset(s, '(', k);
    memset(s + k, ')', k);
    s[2 * k] = '\0';
    char rotulo[32];
    snprintf(rotulo, sizeof rotulo, "%d niveis", k);
    printf("%-18s %s\n", rotulo, balanceada(s) ? "sim" : "nao");
    free(s);
    return 0;
}

O mesmo `balanco_tad.c` compilado com cada implementação. As duas saídas devem ser iguais — e agora
100 000 níveis de parênteses já não transbordam.

In [ ]:
!gcc -Wall -Wextra balanco_tad.c pilha_vetor.c -o balanco_vetor && ./balanco_vetor
!echo ---
!gcc -Wall -Wextra balanco_tad.c pilha_lista.c -o balanco_lista && ./balanco_lista

Experimente o que o tipo incompleto impede: a célula abaixo tenta ler um campo e medir o tamanho de
uma `Pilha` fora de `pilha_vetor.c`. Leia as mensagens do compilador.

In [ ]:
%%writefile espia.c
#include <stdio.h>
#include "pilha.h"

int main(void)
{
    Pilha *p = pilha_cria();
    printf("%d %zu\n", p->n, sizeof(Pilha));
    pilha_libera(p);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra espia.c pilha_vetor.c -o espia 2>&1 | grep error

### Exercício 4 — de decimal para outra base

Os dígitos de `n` na base `b` saem, pelos restos `n % b`, do menos significativo para o mais — ao
contrário da ordem em que se escreve. Uma pilha resolve. Escreva `imprime_base` usando **só** as
operações de `pilha.h`. O teste liga o seu código à versão em **lista** e roda com
`-fsanitize=address`: no Linux, ele também acusa memória que não foi liberada.

In [ ]:
%%writefile ex4.c
#include <stdio.h>
#include "pilha.h"

/* Exercicio 4: imprime n >= 0 na base b (2 <= b <= 16), usando so as
   operacoes de pilha.h. Os restos n % b saem do digito menos
   significativo para o mais: empilhe-os e desempilhe para imprimir.
   TODO: implemente imprime_base. */
void imprime_base(int n, int b)
{
    const char *digitos = "0123456789ABCDEF";
    /* TODO: implemente aqui */
    (void) n;
    (void) b;
    (void) digitos;
}

int main(void)
{
    int casos[][2] = {{10, 2}, {255, 16}, {0, 2}, {2026, 8}, {1000000, 10}};
    for (int i = 0; i < 5; i++) {
        printf("%d na base %d: ", casos[i][0], casos[i][1]);
        imprime_base(casos[i][0], casos[i][1]);
        printf("\n");
    }
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex4.c pilha_lista.c -o ex4
testa('./ex4', '''10 na base 2: 1010
255 na base 16: FF
0 na base 2: 0
2026 na base 8: 3752
1000000 na base 10: 1000000''')

## 5. Quanto custa

Nas três versões — vetor fixo, vetor que dobra, lista —, toda operação de pilha tem custo fixo,
independente de $n$. Mas custo fixo não quer dizer custo igual. `vetor_vs_lista.c` empilha $n$ valores
e desempilha todos, nas duas implementações.

In [ ]:
%%writefile vetor_vs_lista.c
/* vetor_vs_lista.c --- empilhar n valores e desempilhar todos:
   pilha em vetor que dobra x pilha em lista encadeada */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static double agora(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + t.tv_nsec / 1e9;
}

static void *aloca(void *antigo, size_t tam)
{
    void *p = realloc(antigo, tam);
    if (p == NULL) {
        perror("realloc");
        exit(1);
    }
    return p;
}

/* --- vetor que dobra --- */
typedef struct {
    int *v;
    int n, cap;
} PilhaV;

static void empilha_v(PilhaV *p, int x)
{
    if (p->n == p->cap) {
        p->cap = (p->cap == 0) ? 4 : 2 * p->cap;
        p->v = aloca(p->v, p->cap * sizeof *p->v);
    }
    p->v[p->n++] = x;
}

static int desempilha_v(PilhaV *p)
{
    return p->v[--p->n];
}

/* --- lista encadeada: o topo e o primeiro no --- */
typedef struct no {
    int valor;
    struct no *prox;
} No;

static void empilha_l(No **topo, int x)
{
    No *novo = aloca(NULL, sizeof *novo);
    novo->valor = x;
    novo->prox = *topo;
    *topo = novo;
}

static int desempilha_l(No **topo)
{
    No *velho = *topo;
    int x = velho->valor;
    *topo = velho->prox;
    free(velho);
    return x;
}

int main(int argc, char *argv[])
{
    int n = (argc > 1) ? atoi(argv[1]) : 1000000;
    long long soma_v = 0, soma_l = 0;

    double t0 = agora();
    PilhaV pv = {NULL, 0, 0};
    for (int i = 0; i < n; i++)
        empilha_v(&pv, i);
    while (pv.n > 0)
        soma_v += desempilha_v(&pv);
    free(pv.v);
    double t1 = agora();

    No *topo = NULL;
    for (int i = 0; i < n; i++)
        empilha_l(&topo, i);
    while (topo != NULL)
        soma_l += desempilha_l(&topo);
    double t2 = agora();

    printf("n = %d: vetor %.4f s  lista %.4f s  (somas %s)\n", n, t1 - t0,
           t2 - t1, soma_v == soma_l ? "iguais" : "DIFERENTES");
    printf("bytes por elemento: vetor %zu (ate o dobro, se meio vazio)"
           "  lista %zu + o que o malloc gasta por bloco\n",
           sizeof(int), sizeof(No));
    return 0;
}

In [ ]:
# Visualização: empilhar e desempilhar n valores, vetor que dobra x lista
!gcc -O2 -Wall -Wextra vetor_vs_lista.c -o vetor_vs_lista

ns, tv, tl = [], [], []
for n in [1000000, 2000000, 4000000, 6000000, 8000000]:
    s = subprocess.run(['./vetor_vs_lista', str(n)], capture_output=True, text=True).stdout
    p = s.split('\n')[0].split()   # n = N: vetor TV s  lista TL s ...
    ns.append(n)
    tv.append(float(p[4]))
    tl.append(float(p[7]))
print(s.split('\n')[1])

plt.plot(ns, tv, 's-', label='vetor que dobra')
plt.plot(ns, tl, 'o-', label='lista encadeada')
plt.xlabel('n (valores empilhados e desempilhados)')
plt.ylabel('tempo (s)')
plt.title('Pilha: vetor x lista')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

As duas curvas são retas — custo fixo por operação —, mas a da lista é bem mais inclinada. Na lista,
cada `empilha` é um `malloc` e cada `desempilha` um `free`, e os nós ficam espalhados pela memória; no
vetor, os valores estão lado a lado e o `realloc` é raro. A lista gasta ainda 16 bytes por `int` (mais
o que o `malloc` reserva por bloco), contra 4 a 8 do vetor.

| | vetor fixo | vetor que dobra | lista encadeada |
|---|---|---|---|
| `empilha` | constante | constante (amortizado) | constante + `malloc` |
| `desempilha`, `topo`, `vazia` | constante | constante | constante |
| pode encher? | sim, em `MAX` | só sem memória | só sem memória |
| memória por `int` | `MAX` × 4 B | 4 a 8 B | 16 B + o do `malloc` |

### Exercício 5 — calculadora pós-fixa

Na notação **pós-fixa**, o operador vem depois dos operandos: `3 4 + 2 *` é $(3 + 4) \times 2 = 14$.
Ela dispensa parênteses e se avalia com uma pilha de valores: número, empilha; operador, desempilha
dois, calcula e empilha o resultado. No fim, deve sobrar **exatamente um** valor.

Escreva `avalia` usando `pilha.h` (ligado à versão em vetor). Atenção à ordem: em `2 3 -`, o primeiro
valor desempilhado é o 3, o operando da **direita**. Expressões mal formadas e divisão por zero
devolvem 0 — sem *segmentation fault* e sem vazar memória.

In [ ]:
%%writefile ex5.c
#include <stdio.h>
#include <ctype.h>
#include "pilha.h"

/* Exercicio 5: avalia uma expressao pos-fixa com inteiros nao negativos e
   + - * /, separados por espacos. Devolve 1 e o valor em *res, ou 0 se a
   expressao estiver mal formada (falta operando, sobra valor, nada na
   entrada) ou houver divisao por zero. Nao deixe vazar memoria.
   TODO: implemente avalia. */
int avalia(const char *s, int *res)
{
    /* TODO: implemente aqui */
    (void) s;
    (void) res;
    return 0;
}

int main(void)
{
    const char *testes[] = {"3 4 + 2 *", "5 1 2 + 4 * + 3 -", "2 3 -", "8 2 /",
                            "42", "1 +", "1 2", "4 0 /", ""};
    for (int i = 0; i < 9; i++) {
        int v;
        if (avalia(testes[i], &v))
            printf("%-20s = %d\n", testes[i], v);
        else
            printf("%-20s erro\n", testes[i]);
    }
    return 0;
}

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra ex5.c pilha_vetor.c -o ex5
testa('./ex5', '''3 4 + 2 *            = 14
5 1 2 + 4 * + 3 -    = 14
2 3 -                = -1
8 2 /                = 4
42                   = 42
1 +                  erro
1 2                  erro
4 0 /                erro
                     erro''')

## Tarefas: altere o programa da aula

Volte à célula de `balanco.c`, faça cada alteração, reexecute e confira se a saída bate com o que você
previu.

1. **Onde está o erro?** Faça `balanceada` devolver a **posição** do primeiro erro (ou $-1$ se estiver
   tudo certo) e imprima-a ao lado de `nao`. Em `([)]` é fácil; em `(()`, que posição faz sentido
   informar — o fim da cadeia ou o `(` que ficou aberto?
2. **Guardar posições.** Troque a pilha de `char` por uma de `int` que guarda a *posição* de cada
   aberto. O que `combina` passa a receber? Agora dá para imprimir `(()` com uma seta embaixo do `(`
   que sobrou.
3. **Uma pilha pequena.** Troque `MAX` por 4 em `transborda.c` e acrescente a `empilha` um teste que
   encerra o programa com uma mensagem (`fprintf(stderr, ...)` e `exit(1)`) se a pilha estiver cheia.
   Preveja cada linha da saída antes de executar.

## Desafio — uma calculadora de verdade

Junte as peças: leia expressões **infixas** — inteiros não negativos, `+ - * /`, parênteses e espaços,
uma por linha — e imprima o valor ou uma mensagem de erro.

1. Converta para pós-fixa com uma pilha de **operadores**, varrendo a expressão da esquerda para a
   direita: número vai direto para a saída; `(` é empilhado; `)` desempilha para a saída até achar o
   `(` (se não achar, os parênteses estão errados); um operador primeiro desempilha para a saída os
   operadores de precedência **maior ou igual** que estão no topo (até um `(`) e só então é empilhado.
   No fim, desempilhe o que sobrou — um `(` aí também é erro.
2. Avalie a pós-fixa com uma pilha de **valores**, como no Exercício 5.

O "maior ou igual" faz `10 - 4 - 3` dar 3, e não 9. Use só `pilha.h`; o teste usa a versão em vetor e
`-fsanitize=address`.

In [ ]:
%%writefile calc.c
#include <stdio.h>
#include <ctype.h>
#include <string.h>
#include "pilha.h"

enum { OK, ERRO_PARENTESES, ERRO_DIVISAO, ERRO_EXPRESSAO };

/* Desafio: calcula a expressao infixa s (inteiros nao negativos, + - * /,
   parenteses e espacos). Devolve OK e o valor em *res, ou um dos codigos
   de erro. Sugestao: com uma pilha de operadores, gere a pos-fixa numa
   cadeia; depois avalie-a com uma pilha de valores (Exercicio 5).
   Operadores de mesma precedencia sao calculados da esquerda para a direita.
   TODO: implemente calcula. */
int calcula(const char *s, int *res)
{
    /* TODO: implemente aqui */
    (void) s;
    (void) res;
    return ERRO_EXPRESSAO;
}

int main(void)
{
    char linha[256];
    const char *msg[] = {"", "erro: parenteses", "erro: divisao por zero",
                         "erro: expressao"};
    while (fgets(linha, sizeof linha, stdin) != NULL) {
        linha[strcspn(linha, "\n")] = '\0';
        int v, r = calcula(linha, &v);
        if (r == OK)
            printf("%d\n", v);
        else
            printf("%s\n", msg[r]);
    }
    return 0;
}

In [ ]:
entrada = '''2 * (3 + 4) - 5
10 - 4 - 3
2 + 3 * 4
100 / 7 / 2
((7))
(1 + 2
1 + 2)
8 / (4 - 4)
3 +
'''

In [ ]:
!gcc -g -fsanitize=address -Wall -Wextra calc.c pilha_vetor.c -o calc
testa('./calc', '''9
3
14
7
7
erro: parenteses
erro: parenteses
erro: divisao por zero
erro: expressao''', entrada=entrada)

Depois de passar no teste, experimente: o que acontece com `2 * -3`? E com um número maior que
`INT_MAX`? Que mudanças seriam necessárias para aceitar o menos unário e o operador `^` (potência), que
associa da **direita** para a esquerda (`2 ^ 3 ^ 2` = $2^9$)?

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. Os pontos principais desta aula estão em:

- Celes, Cerqueira e Rangel (2004), *Introdução a estruturas de dados*, cap. 11 — pilhas em vetor e em
  lista; cap. 9 — tipos abstratos de dados;
- Cormen et al. (2009), *Introduction to Algorithms*, §10.1 — pilhas (`PUSH`, `POP`, *underflow* e
  *overflow*); §17.4 — tabelas dinâmicas e o custo amortizado de dobrar;
- Knuth (1997), *The Art of Computer Programming*, vol. 1, §2.2.1 — pilhas, filas e deques; §2.2.2 —
  alocação sequencial, inclusive duas pilhas num vetor;
- Feofiloff (2009), *Algoritmos em linguagem C*, cap. 6 — pilhas, parênteses e notação posfixa;
- Tenenbaum, Langsam e Augenstein (1995), *Estruturas de dados usando C*, cap. 2 — a pilha e a
  conversão infixa → pós-fixa;
- Dijkstra (1961) — o algoritmo de conversão com pilha de operadores;
- Serebryany et al. (2012) — o AddressSanitizer.